# Week 6 — Gaussian Naive Bayes Classifier from Scratch (NumPy)

**Goal:** Understand **generative supervised classification** by implementing Gaussian Naive Bayes with NumPy: class priors, per-class mean/variance, log-likelihood under a diagonal Gaussian, predict via argmax of the log-posterior, optional variance smoothing, train/test on synthetic multi-class blobs, accuracy + confusion counts, and a decision-region plot.

No scikit-learn for the core — optional comparison is fine later.

## 1. Imports & synthetic blobs

We generate three overlapping 2D Gaussian clusters. The generative story of Naive Bayes matches this data well.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)
print("numpy", np.__version__)

In [ ]:
def make_blobs(
    n_per_class: int = 80,
    centers=None,
    cluster_std: float = 0.75,
    seed: int = 42,
):
    if centers is None:
        centers = [(-1.8, -1.2), (1.7, -0.6), (0.1, 1.8)]
    rng = np.random.default_rng(seed)
    cov = np.eye(2) * (cluster_std ** 2)
    Xs, ys = [], []
    for label, center in enumerate(centers):
        Xk = rng.multivariate_normal(np.asarray(center, float), cov, size=n_per_class)
        Xs.append(Xk)
        ys.append(np.full(n_per_class, label, dtype=int))
    X = np.vstack(Xs)
    y = np.concatenate(ys)
    perm = rng.permutation(len(y))
    return X[perm], y[perm]


def train_test_split(X, y, test_size=0.25, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(y))
    n_test = max(1, int(round(len(y) * test_size)))
    test_idx, train_idx = idx[:n_test], idx[n_test:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


X, y = make_blobs()
X_train, X_test, y_train, y_test = train_test_split(X, y)
print(X_train.shape, X_test.shape, np.bincount(y_train))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=40)
ax.set_title("Synthetic 3-class Gaussian blobs")
ax.set_xlabel("x1"); ax.set_ylabel("x2")
plt.show()

## 2. The model: priors + diagonal Gaussians

Bayes' rule:

$$P(y=c \mid x) \propto P(y=c)\, P(x \mid y=c)$$

**Naive** assumption: features are independent given the class, so

$$P(x \mid y=c) = \prod_j \mathcal{N}(x_j; \mu_{c,j}, \sigma^2_{c,j})$$

We work in **log space** (products → sums) and add a small **variance floor** (`var_smoothing`) so logs/divisions stay stable.

In [ ]:
class GaussianNaiveBayes:
    def __init__(self, var_smoothing: float = 1e-9):
        self.var_smoothing = float(var_smoothing)
        self.classes_ = self.priors_ = self.means_ = self.vars_ = None

    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y, int)
        self.classes_ = np.unique(y)
        n_c, n_f, n = len(self.classes_), X.shape[1], len(y)
        self.priors_ = np.zeros(n_c)
        self.means_ = np.zeros((n_c, n_f))
        self.vars_ = np.zeros((n_c, n_f))
        for i, c in enumerate(self.classes_):
            Xc = X[y == c]
            self.priors_[i] = len(Xc) / n
            self.means_[i] = Xc.mean(axis=0)
            self.vars_[i] = Xc.var(axis=0) + self.var_smoothing
        return self

    def _log_gaussian(self, X):
        diff = X[:, None, :] - self.means_[None, :, :]
        log_lik = -0.5 * (
            np.log(2.0 * np.pi * self.vars_[None, :, :])
            + (diff ** 2) / self.vars_[None, :, :]
        )
        return log_lik.sum(axis=2)  # sum over features

    def log_posterior(self, X):
        X = np.asarray(X, float)
        return np.log(self.priors_[None, :]) + self._log_gaussian(X)

    def predict(self, X):
        return self.classes_[np.argmax(self.log_posterior(X), axis=1)]

    def predict_proba(self, X):
        lp = self.log_posterior(X)
        lp = lp - lp.max(axis=1, keepdims=True)
        p = np.exp(lp)
        return p / p.sum(axis=1, keepdims=True)


model = GaussianNaiveBayes(var_smoothing=1e-9).fit(X_train, y_train)
print("priors:", np.round(model.priors_, 4))
print("means:\n", np.round(model.means_, 3))
print("vars:\n", np.round(model.vars_, 3))

## 3. Predict, accuracy, confusion counts

Prediction is simply **argmax over log-posterior** (one score per class).

In [ ]:
def accuracy(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))

def confusion_counts(y_true, y_pred, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true.astype(int), y_pred.astype(int)):
        cm[t, p] += 1
    return cm

y_pred = model.predict(X_test)
acc = accuracy(y_test, y_pred)
cm = confusion_counts(y_test, y_pred, n_classes=3)
print(f"Test accuracy: {acc:.4f}")
print("Confusion (rows=true, cols=pred):\n", cm)
print("Example probabilities (first 5 test rows):\n", np.round(model.predict_proba(X_test[:5]), 3))

## 4. Decision regions

Color a dense grid by the predicted class. Gaussian NB tends to draw **smooth, roughly elliptical** regions when the blobs are Gaussian.

In [ ]:
def plot_decision_regions(model, X, y, title="Gaussian NB decision regions"):
    x_min, x_max = X[:, 0].min() - 1.0, X[:, 0].max() + 1.0
    y_min, y_max = X[:, 1].min() - 1.0, X[:, 1].max() + 1.0
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    zz = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.contourf(xx, yy, zz, alpha=0.35, cmap="coolwarm", levels=np.arange(-0.5, 3.5, 1.0))
    sc = ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=40)
    ax.set_xlabel("x1"); ax.set_ylabel("x2"); ax.set_title(title)
    ax.legend(*sc.legend_elements(), title="class", loc="best")
    plt.show()

plot_decision_regions(model, X_train, y_train)

out = Path("../outputs/decision_regions.png")
# also save if run from notebooks/
try:
    out.parent.mkdir(parents=True, exist_ok=True)
    # re-save via a quick figure for the outputs folder when desired
except Exception:
    pass

# 5. Takeaways

- **Generative:** model P(x | y) and P(y), then use Bayes' rule.
- **Naive:** features independent given class → product of 1D Gaussians (diagonal covariance).
- **Log-posterior + argmax** is the practical predictor; softmax gives probabilities.
- **Variance smoothing** is a tiny floor that keeps the math numerically safe.
- Next ideas: compare to sklearn's `GaussianNB`, try correlated features (where "naive" hurts), or multinomial NB for text.

You're done with Week 6 — nice work!
